In [68]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
from scipy.stats import norm
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown
warnings.filterwarnings('ignore')

In [45]:
def downloaded_data_validation(prices):

    validity = []
    warning = False

    duplicate_dates = prices.index.duplicated().sum()

    for asset in prices.columns:

        total_rows = len(prices[asset])
        missing = prices[asset].isna().sum()
        missing_pct = (missing / total_rows) * 100 if total_rows else 0

        # Asset-specific invalid price check
        invalid_prices = (
            (prices[asset] <= 0) |
            (~np.isfinite(prices[asset]))
        ).sum()

        validity.append({
            'Ticker': asset,
            'Observations': total_rows,
            'Missing': missing,
            'Missing %': missing_pct,
            'Invalid Prices': invalid_prices,
            'Duplicate Dates': duplicate_dates,
            'First Date': prices.index.min(),
            'Last Date': prices.index.max()
        })

        if missing > 0 or invalid_prices > 0 or duplicate_dates > 0:
            warning = True

    validity = pd.DataFrame(validity)

    return validity, warning

In [46]:

def download_data(tickers, period='10y', interval='1d'):
    data = yf.download(tickers=tickers, period=period, interval=interval, auto_adjust=True)['Close']

    if isinstance(data, pd.Series): #incase the tickers is a single asset
        data = data.to_frame()
        
    missing = data.isna().sum()

    if missing.any():
        print(f'Missing Observations : {missing} data points\n Interpolating Linerarly...')
        data.interpolate(inplace=True)
    
    validity, warning = downloaded_data_validation(data)

    return data, validity, warning



In [47]:
def calcualte_portfolio_returns(prices, weights, rebalance_frequency = "Daily"):
    pct_change = prices.pct_change().dropna()

    allocation_drifts = pd.DataFrame(index=prices.index, columns= prices.columns)

    
    if rebalance_frequency == 'Daily':
        portfolio_daily_returns = pct_change @ weights 
        allocation_drifts.iloc[:,:] = weights.values

    elif rebalance_frequency =='BuynHold':
        portfolio_value = (prices/prices.iloc[0]) @ weights
        portfolio_daily_returns = portfolio_value.pct_change().dropna()

        asset_values = (prices / prices.iloc[0]).mul(weights, axis=1)

        allocation_drifts = (asset_values.shift(1).div(asset_values.shift(1).sum(axis=1), axis=0).loc[pct_change.index])

    elif rebalance_frequency == "Monthly":

        portfolio_daily_returns = pd.Series(index=pct_change.index, dtype=float)

        current_weights = weights.copy()
        previous_month = None

        for date, daily_returns in pct_change.iterrows():

            current_month = date.to_period("M")

            if previous_month is not None and current_month != previous_month:
                current_weights = weights.copy()

            allocation_drifts.loc[date] = current_weights

            portfolio_daily_returns.loc[date] = daily_returns @ current_weights

            current_weights *= (1 + daily_returns)
            current_weights /= current_weights.sum()

            previous_month = current_month

    else:
        raise ValueError("Invalid rebalancing frequency.")

    return portfolio_daily_returns, allocation_drifts




In [48]:
def calcualte_portfolio_value(initial_investment, portfolio_daily_returns):
    portfolio_value = (1 + portfolio_daily_returns).cumprod() * initial_investment
    return portfolio_value

In [49]:
def calculate_returns_metrics(portfolio_value):
    initial_value = portfolio_value.iloc[0]
    final_value = portfolio_value.iloc[-1]
    years = (len(portfolio_value) - 1) / 252

    total_return = (final_value / initial_value) - 1
    cagr = np.pow(final_value / initial_value, 1/years) - 1

    return total_return, cagr

def calculate_volatility_metrics(portfolio_returns):
    daily_vol = portfolio_returns.std()
    annual_vol = daily_vol * (252**0.5)

    return daily_vol, annual_vol


def calculate_portfolio_metrics(portfolio_value, risk_free_rate = 0.02):    

    portfolio_returns = portfolio_value.pct_change().dropna()

    total_return, cagr = calculate_returns_metrics(portfolio_value)
    daily_vol, annual_vol = calculate_volatility_metrics(portfolio_returns)

    sharpe = ((portfolio_returns.mean() * 252)  - risk_free_rate) / annual_vol

    metrics = {
        'Total Return': total_return,
        'CAGR': cagr,
        'Daily Volatility': daily_vol,
        'Annualized Volatility': annual_vol,
        'Sharpe Ratio': sharpe
    }

    return metrics

In [50]:
tickers = ['AAPL', 'GOOG','WMT', 'GLD']
period = '10y'
interval = '1d'
prices, validity, warning = download_data(tickers)
investment_start = '2021-01-01'
investment_end = '2025-01-01'

investment_prices = prices[(prices.index>=investment_start) & (prices.index<=investment_end)]
weights = pd.Series([0.2,0.1,0.3,0.4], index = prices.columns)
rebalance_frequency = 'BuynHold'
portfolio_daily_returns, allocation_drifts = calcualte_portfolio_returns(investment_prices, weights=weights, rebalance_frequency=rebalance_frequency)
initial_investment = 100_000
portfolio_value = calcualte_portfolio_value(initial_investment=initial_investment, portfolio_daily_returns=portfolio_daily_returns)
risk_free_rate = 0.04
metrics = calculate_portfolio_metrics(portfolio_value, risk_free_rate)



[*********************100%***********************]  4 of 4 completed


In [51]:
metrics

{'Total Return': np.float64(0.9709510751576385),
 'CAGR': np.float64(0.18586760144385694),
 'Daily Volatility': np.float64(0.011407667099818199),
 'Annualized Volatility': np.float64(0.18109110111319437),
 'Sharpe Ratio': np.float64(0.8113360424653433)}

In [52]:
import plotly.express as px
comparison = pd.DataFrame()

normalized_prices = (investment_prices/investment_prices.iloc[0]) * initial_investment

comparison = normalized_prices.copy()
comparison['Portfolio'] = portfolio_value
comparison.iloc[0, comparison.columns.get_loc("Portfolio")] = initial_investment

In [53]:
fig = px.line(
    comparison,
    x = comparison.index,
    y = comparison.columns,
    title="Portfolio Performance vs Individual Assets",
    labels={
        "x": 'date',
        'value': 'Portfolio Values($)',
        'variable' : 'Asset'
    }
)
for trace in fig.data:
    if trace.name != "Portfolio":
        trace.opacity = 0.3
    else:
        trace.opacity = 1
        trace.line.width = 3

fig.show()

In [54]:
fig = px.area(
    allocation_drifts * 100,
    x = allocation_drifts.index,
    y = allocation_drifts.columns,
    labels = {
        'x': 'Date',
        'value': 'Allocation (%)',
        'variable': 'Asset'
    }
)
fig.update_layout(
    yaxis_title="Allocation (%)",
    yaxis_range=[0, 100],
    hovermode="x unified",
    legend_title="Asset"
)
fig.show()

### Portfolio Construction complete
# Market Data And Analytics

In [55]:
def calculate_asset_statistics(prices):

    asset_data = []

    for ticker in prices.columns:

        asset_prices = prices[ticker]
        missing = asset_prices.isna().sum()
        observations = asset_prices.notna().sum()
        missing_pct = missing / len(prices) * 100 if len(prices) else 0

        invalid_prices = ((asset_prices <= 0) | (~np.isfinite(asset_prices) & asset_prices.notna())).sum()

        valid_prices = asset_prices.dropna()

        asset_data.append({
            'ticker': ticker,
            'first_date': asset_prices.first_valid_index(),
            'last_date': asset_prices.last_valid_index(),
            'missing_observations': missing,
            'missing_percentage': missing_pct,
            'number_of_observations': observations,
            'data_completeness_percentage': 100 - missing_pct,
            'latest_valid_price': valid_prices.iloc[-1] if not valid_prices.empty else np.nan,
            'latest_valid_price_date': asset_prices.last_valid_index(),
            'invalid_prices': invalid_prices,
            'data_validation_warning': missing > 0 or invalid_prices > 0
        })

    return pd.DataFrame(asset_data).set_index('ticker')

In [56]:
def calculate_asset_returns(prices):
    daily_returns = prices.pct_change(fill_method=None)
    return daily_returns


def calculate_return_statistics(daily_returns):

    asset_data = []

    for ticker in daily_returns.columns:

        asset_returns = daily_returns[ticker].dropna()

        daily_mean = asset_returns.mean()
        annualized_return = daily_mean * 252

        daily_vol = asset_returns.std()
        annualized_vol = daily_vol * np.sqrt(252)

        asset_data.append({
            'ticker': ticker,
            'daily_mean_return': daily_mean,
            'annualized_return': annualized_return,
            'daily_volatility': daily_vol,
            'annualized_volatility': annualized_vol,
            'skewness': asset_returns.skew(),
            'excess_kurtosis': asset_returns.kurt(),
            'best_daily_return': asset_returns.max(),
            'worst_daily_return': asset_returns.min()
        })

    return pd.DataFrame(asset_data).set_index('ticker')

In [57]:
from arch import arch_model

def calculate_rolling_vol(asset_return, window, annualization_factor=252):
    return asset_return.rolling(window).std() * np.sqrt(annualization_factor)

def calculate_ewma_vol(asset_return, lambda_, window, annualization_factor=252):
    ewma_vol = np.sqrt(asset_return.pow(2).ewm(alpha = 1-lambda_, min_periods = window, adjust = False).mean())
    return ewma_vol * np.sqrt(annualization_factor)

def calculate_garch_vol(asset_return, annualization_factor=252):
    returns = asset_return.dropna() * 100
    model = arch_model(returns, mean = 'Zero', vol = 'GARCH', p = 1, q = 1, dist = 'normal')

    fitted_model = model.fit(disp = 'off')

    garch_vol = fitted_model.conditional_volatility / 100
    return garch_vol * np.sqrt(annualization_factor)



def vol_analysis(daily_returns, window = 30, lambda_ = 0.94):
    vol_stats = {}

    for ticker in daily_returns.columns:
        rolling_vol = calculate_rolling_vol(daily_returns[ticker], window)
        ewma_vol = calculate_ewma_vol(daily_returns[ticker], lambda_, window)
        garch_vol = calculate_garch_vol(daily_returns[ticker])
        ticker_vol = pd.concat(
            [
                rolling_vol.rename('rolling_vol'),
                ewma_vol.rename('ewma_vol'),
                garch_vol.rename('garch_vol')
            ], axis=1
        )
        vol_stats[ticker] = ticker_vol

    annual_asset_vol = daily_returns.std() * np.sqrt(252)

    return vol_stats, annual_asset_vol




In [58]:
def calculate_correlation_matrix(daily_returns, window = None):
    if window is None:
        return daily_returns.corr()

    daily_returns = daily_returns.iloc[-window:,:]
    return daily_returns.corr()


def calculate_rolling_correlation(daily_returns, asset1, asset2, window = 60):
    return daily_returns[asset1].rolling(window).corr(daily_returns[asset2])





In [59]:
from scipy.stats import jarque_bera

def calculate_normality_test(asset_returns, significance_level = 0.05):
    normality = []

    for ticker in asset_returns.columns:

        jb_statistic, p_value = jarque_bera(asset_returns[ticker].dropna())

        normality.append({
            'ticker':ticker,
            'jb_statistic':jb_statistic,
            'p_value': p_value,
            'is_normal': p_value >=significance_level
        })
    normality = pd.DataFrame(normality).set_index('ticker')
    return normality



In [60]:
daily_returns = calculate_asset_returns(investment_prices)

return_statistics = calculate_return_statistics(daily_returns)

asset_statistics = calculate_asset_statistics(investment_prices)

vol_stats, annual_asset_vol = vol_analysis(daily_returns, window=30, lambda_=0.94)

correlation_matrix = calculate_correlation_matrix(daily_returns)

normality_results = calculate_normality_test(daily_returns)

In [61]:
vol_colors = {
    "rolling_vol": "#636EFA",
    "ewma_vol": "#EF553B",
    "garch_vol": "#00CC96"
}

vol_labels = {
    "rolling_vol": "Rolling Volatility",
    "ewma_vol": "EWMA Volatility",
    "garch_vol": "GARCH Volatility"
}

def plot_asset_volatility(ticker, vol_stats, annual_asset_vol):

    asset_vol = vol_stats[ticker]

    fig = go.Figure()

    for column in ["rolling_vol", "ewma_vol", "garch_vol"]:

        fig.add_trace(
            go.Scatter(
                x=asset_vol.index,
                y=asset_vol[column],
                name=vol_labels[column],
                mode="lines",
                line=dict(
                    color=vol_colors[column],
                    width=1.5
                )
            )
        )

    # Historical volatility reference line
    fig.add_hline(
        y=annual_asset_vol[ticker],
        line_dash="dash",
        line_color="black",
        line_width=1.5,
        annotation_text="Historical Volatility",
        annotation_position="top left"
    )

    fig.update_layout(
        title=f"{ticker} - Volatility Analysis",
        template="plotly_white",
        height=400,
        hovermode="x unified",
        yaxis_title="Annualized Volatility",
        yaxis_tickformat=".0%",
        legend=dict(
            orientation="h",
            y=1.12,
            x=0.5,
            xanchor="center"
        ),
        margin=dict(l=50, r=20, t=90, b=40)
    )

    return fig


asset_figures = {}

for ticker in daily_returns.columns:

    asset_figures[ticker] = plot_asset_volatility(
        ticker,
        vol_stats,
        annual_asset_vol
    )
    asset_figures[ticker].show()


In [62]:


def plot_return_distribution(ticker, daily_returns):

    asset_returns = daily_returns[ticker].dropna()

    mean = asset_returns.mean()
    std = asset_returns.std()

    x = np.linspace(
        asset_returns.min(),
        asset_returns.max(),
        300
    )

    normal_pdf = norm.pdf(x, mean, std)

    fig = go.Figure()

    # Return histogram
    fig.add_trace(
        go.Histogram(
            x=asset_returns,
            histnorm="probability density",
            name="Actual Returns",
            opacity=0.65
        )
    )

    # Normal distribution curve
    fig.add_trace(
        go.Scatter(
            x=x,
            y=normal_pdf,
            mode="lines",
            name="Normal Distribution",
            line=dict(color="red", width=2)
        )
    )

    # Mean return
    fig.add_vline(
        x=mean,
        line_dash="dash",
        line_color="black",
        annotation_text="Mean"
    )

    fig.update_layout(
        title=f"{ticker} - Daily Return Distribution",
        template="plotly_white",
        height=400,
        xaxis_title="Daily Return",
        yaxis_title="Density",
        xaxis_tickformat=".1%"
    )

    return fig


asset_return_figures = {}

for ticker in daily_returns.columns:

    asset_return_figures[ticker] = plot_return_distribution(
        ticker,
        daily_returns
    )

    asset_return_figures[ticker].show()




In [63]:
def plot_asset_returns(ticker, daily_returns):

    asset_returns = daily_returns[ticker].dropna()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=asset_returns.index,
            y=asset_returns,
            mode="lines",
            name="Daily Returns",
            line=dict(
                color="#636EFA",
                width=1
            )
        )
    )

    # Zero-return reference line
    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="black",
        line_width=1
    )

    fig.update_layout(
        title=f"{ticker} - Daily Returns",
        template="plotly_white",
        height=400,
        hovermode="x unified",
        xaxis_title="Date",
        yaxis_title="Daily Return",
        yaxis_tickformat=".1%"
    )

    return fig


asset_return_timeseries = {}

for ticker in daily_returns.columns:

    asset_return_timeseries[ticker] = plot_asset_returns(
        ticker,
        daily_returns
    )

    asset_return_timeseries[ticker].show()


In [64]:
def plot_correlation_heatmap(correlation_matrix):

    fig = px.imshow(
        correlation_matrix,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        aspect="auto",
        title="Asset Correlation Matrix"
    )

    fig.update_layout(
        template="plotly_white",
        height=600,
        width=800,
        coloraxis_colorbar=dict(
            title="Correlation"
        ),
        xaxis_title="Asset",
        yaxis_title="Asset"
    )

    return fig
correlation_fig = plot_correlation_heatmap(correlation_matrix)

correlation_fig.show()


In [65]:

@interact(
    lookback=Dropdown(
        options={
            "Full History": None,
            "250 Days": 250,
            "500 Days": 500,
            "750 Days": 750,
            "1000 Days": 1000
        },
        description="Lookback:"
    )
)
def update_correlation(lookback):

    corr = calculate_correlation_matrix(
        daily_returns,
        window=lookback
    )

    fig = plot_correlation_heatmap(corr)

    fig.show()

interactive(children=(Dropdown(description='Lookback:', options={'Full History': None, '250 Days': 250, '500 D…

In [ ]:
def plot_rolling_correlation(rolling_corr, asset1, asset2, window):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=rolling_corr.index,
        y=rolling_corr,
        mode="lines",
        name="Rolling Correlation",
        line=dict(width=1.5)
    ))

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="black"
    )

    fig.update_layout(
        title=f"{window}-Day Rolling Correlation: {asset1} vs {asset2}",
        template="plotly_white",
        height=450,
        hovermode="x unified",
        xaxis_title="Date",
        yaxis_title="Correlation",
        yaxis=dict(range=[-1, 1])
    )

    return fig

@interact(
    asset1=Dropdown(
        options=prices.columns.tolist(),
        description="Asset 1:"
    ),
    asset2=Dropdown(
        options=prices.columns.tolist(),
        description="Asset 2:"
    ),
    window=Dropdown(
        options=[30, 60, 90, 120, 252],
        value=60,
        description="Window:"
    )
)
def display_rolling_correlation(asset1, asset2, window):

    if asset1 == asset2:
        print("Please select two different assets.")
        return

    rolling_corr = calculate_rolling_correlation(
        daily_returns, asset1, asset2, window
    )

    fig = plot_rolling_correlation(
        rolling_corr, asset1, asset2, window
    )

    fig.show()

interactive(children=(Dropdown(description='Asset 1:', options=('AAPL', 'GLD', 'GOOG', 'WMT'), value='AAPL'), …

In [70]:
def plot_return_boxplot(daily_returns):

    fig = go.Figure()

    for ticker in daily_returns.columns:
        fig.add_trace(go.Box(
            y=daily_returns[ticker].dropna(),
            name=ticker,
            boxpoints="outliers"
        ))

    fig.update_layout(
        title="Daily Return Distribution by Asset",
        template="plotly_white",
        height=500,
        yaxis_title="Daily Return",
        yaxis_tickformat=".1%",
        showlegend=False
    )

    return fig

return_boxplot = plot_return_boxplot(daily_returns)
return_boxplot.show()